# Week 6C: GitHub Pages, Actions and a Weekly Literature Watch

**GGS 662 Agentic GeoAI**

Scenario: you submit a paper and associated code. Three months later, reviews arrive. What related work appeared or changed while you waited, and what should you read before revising?

Build a small scheduled watch that preserves source metadata, optionally assesses relevance, writes a weekly briefing, and accumulates a review-period ledger. Publish a selected static report on GitHub Pages if useful.

**Practical guide; no notebook kernel required.** Terminal commands run in a separate student demonstration repository. Start offline, then test a live collection before enabling a schedule. The sample is deliberately bounded and does not promise coverage of all literature or GitHub developments.

Supporting starter: [week6/literature_watch](week6/literature_watch/). Nothing is scheduled or deployed by opening this notebook.

## 1. GitHub provides more than code storage

| Capability | Purpose | Example |
|---|---|---|
| Git repository | Versioned files and changes | Code, research brief, dated reports |
| Issues / pull requests | Discuss and review work | Human decision about a candidate paper |
| Releases / artifacts | Package outputs | Download run results |
| Pages | Static website hosting | Readable research-watch report |
| Actions | Trigger and run jobs | Weekly literature collection |

Pages serves HTML/CSS/JavaScript; it is not a continuously running Python agent or a place to hide API secrets. Actions runs the collector on a runner; your laptop does not need to remain awake. [GitHub Pages overview](https://docs.github.com/en/pages/getting-started-with-github-pages/what-is-github-pages)

Our sequence is **schedule -> retrieve metadata -> compare saved state -> optional relevance assessment -> reports -> durable commit -> optional static-site deployment**.



## 2. Be precise about the word agent

The starter has three modes:

- `fixture`: synthetic offline example; no network or model use.
- `live`: deterministic arXiv metadata collection and change detection; no model calls.
- `agent`: the same collection followed by at most one bounded model call assessing up to eight changed records.

This is an **AI-assisted scheduled workflow**, with a constrained relevance assessor. It is not an open-ended autonomous scientist. The model does not choose searches or execute arbitrary tools. That tradeoff makes the example inexpensive and auditable.

GitHub Actions is the scheduler/executor, not the language model. A model API key and billing are separate from your desktop ChatGPT/Codex subscription. You can complete the teaching activity without any model API calls.


## 3. Define the paper/codebase being watched

Use a separate demonstration repository. Create a short, human-approved brief from your paper abstract and README: question, contribution, methods, datasets, limitations and relevant keywords. Link your actual paper/codebase if appropriate; the supplied inland-waterway brief is **illustrative**, not a real submitted manuscript.

Do not send the whole repository to the model each week. The runner uses a brief capped at 4,000 characters and abstracts capped at 3,000 characters per record.

Edit `watch_config.json`:

- Set the actual submission date (YYYY-MM-DD).
- Replace project title and brief.
- Set an arXiv query that you can inspect manually.
- Keep classroom bounds: two pages of 25 results; at most eight assessed records; at most 1,600 output tokens.
- Select an available model explicitly only if using `agent` mode.

Source: [arXiv API manual](https://info.arxiv.org/help/api/user-manual.html). The API returns Atom metadata and exposes publication/update timestamps. These are not dates when the researcher discovered the work.

**Exercise:** explain why a publication date, revision date and first-observed date should all be preserved.

## 4. Copy the starter into your own repository

Create an empty repository such as `paper_literature_watch`, clone it and open it locally. For Pages, start with a demonstration project containing material you are willing to publish.

Copy the **contents** of `week6/literature_watch/` into the new repository root. Then move `workflow_templates/literature-watch.yml` to `.github/workflows/literature-watch.yml`. Use the editor or a coding agent:

```text
Copy the course literature_watch starter into my demonstration repository.
Put watch.py, watch_config.json and test_watch.py at its root.
Place the supplied YAML in .github/workflows/literature-watch.yml.
Do not enable live API triage or publish Pages yet.
Do not copy unrelated course files or credentials.
```

The template lives outside the course's .github/workflows directory deliberately: adding this teaching material does not activate a scheduled job in the course repository.

Generated structure:

```text
watch.py / watch_config.json / test_watch.py
.github/workflows/literature-watch.yml
state/       durable signatures, events and run history
runs/        per-run metadata, events and briefing
reports/     latest.md and review_period.md
docs/        static index.html for Pages
```

## 5. Run the offline mini-example first

From the new repository root:

```text
python watch.py --mode fixture --output demo_output --today 2026-10-05
python watch.py --mode fixture --output demo_output --today 2026-10-05
python -m unittest test_watch.py
```

Use `python3` if that is your configured macOS command. No dependencies or keys are required.

Open `demo_output/reports/latest.md`, `demo_output/reports/review_period.md`, and `demo_output/docs/index.html` in your editor/browser. The invented DEMO-001 record is explicitly marked synthetic. On the second run it should not create a duplicate event. Fixture state is separate from live state.

**Checkpoint:** an offline report exists, the repeat run contains zero new events, and the review-period ledger still contains the original observation. This tests execution, not scientific usefulness.

The offline tests also verify that changing the watch scope fails rather than silently merging incompatible histories.

## 6. Inspect a live metadata run

```text
python watch.py --mode live
```

Review the query/results and false positives. This makes at most two arXiv requests with a delay between pages. Network/API failures stop the run; they are not reported as a successful empty search.

The first run looks back to submission. Later runs overlap the last successful search by 14 days to catch some delayed indexing and revisions. IDs, content signatures and timestamps identify new or changed observations.

A hard result cap can truncate coverage. When capped, the watermark does not advance; inspect the coverage warning. Repeating a capped query will not magically reach older pages. Narrow the query or perform a reviewed backfill in a separate output directory. Changing the scope requires a separate history or explicit migration.

arXiv is not a complete journal index. This does not monitor code releases, PRs, citation counts or all scientific events. Treat it as a starter watch and add separately tested sources later. Full texts are not reviewed.

## 7. Optional bounded relevance assessment

For `agent` mode, obtain an OpenAI API key and set `OPENAI_API_KEY` through a secure environment or GitHub secret. Do not put it in a notebook or configuration file. Choose a supported model ID in `watch_config.json`.

```text
python watch.py --mode agent
```

The runner uses the Responses API, records returned usage, and requests a short source-ID-based briefing from abstract metadata. There are no model tools, follow-up calls or automatic retries. Incomplete model responses fail without advancing saved state. [Official text-generation guidance](https://developers.openai.com/api/docs/guides/text)

Configure model/provider spending controls. An output-token limit bounds response generation, not the entire bill. Input, model-specific billing and GitHub runner usage also matter.

If more than eight events arrive, the remainder stays visible in the ledger but is not assessed. Previously collected metadata-only events are not automatically assessed on a later agent run: review them locally or start a separate scoped backfill. No-change runs make zero model calls.

**Human audit:** select one assessed paper; check its abstract, relevance judgment and uncertainty. A generated summary cannot establish novelty or replace reading the paper.

## 8. Understand the workflow YAML

The workflow supports manual dispatch and a Monday 13:17 UTC schedule. UTC avoids assuming a fixed local clock across daylight-saving changes.

Scheduled runs require the workflow on the default branch, can be delayed, and public-repository inactivity can disable schedules. Check run history rather than assuming exact weekly execution. [GitHub workflow triggers](https://docs.github.com/en/actions/reference/workflows-and-actions/events-that-trigger-workflows)

Each runner checks out the durable state from Git, runs the collector, saves an artifact and commits changed reports/state. The watch job needs `contents: write`; a concurrency group prevents overlapping watch runs. Branch protections or concurrent human changes can reject a push: inspect failure and recover from the artifact, rather than force-push.

Action versions are tagged for readability. For a production deployment, pin reviewed action revisions according to your organization's policy.

The complete supplied YAML follows; use it in your own demonstration repository.

```yaml
name: Weekly literature watch
on:
  schedule:
    - cron: '17 13 * * 1'
  workflow_dispatch:
    inputs:
      mode:
        description: 'fixture (offline), live (metadata), or agent (paid API triage)'
        required: true
        default: fixture
        type: choice
        options: [fixture, live, agent]
permissions:
  contents: read
concurrency:
  group: literature-watch
  cancel-in-progress: false
jobs:
  watch:
    runs-on: ubuntu-latest
    timeout-minutes: 10
    permissions:
      contents: write
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - name: Run collector and optional agent
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          WATCH_MODE: ${{ github.event_name == 'schedule' && vars.WATCH_MODE || inputs.mode }}
        run: |
          python watch.py --mode "${WATCH_MODE:-live}"
      - uses: actions/upload-artifact@v4
        with:
          name: literature-watch-${{ github.run_id }}
          path: |
            reports/
            docs/
            state/
            runs/
          retention-days: 90

      - name: Commit durable state and reports
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "41898282+github-actions[bot]@users.noreply.github.com"
          git add state reports runs docs
          if ! git diff --cached --quiet; then
            git commit -m "Update bounded literature watch"
            git push
          fi
      - name: Upload optional Pages site
        if: vars.PUBLISH_PAGES == 'true'
        uses: actions/upload-pages-artifact@v3
        with:
          path: docs
  pages:
    if: vars.PUBLISH_PAGES == 'true'
    needs: watch
    runs-on: ubuntu-latest
    permissions:
      pages: write
      id-token: write
    environment:
      name: github-pages
      url: ${{ steps.deployment.outputs.page_url }}
    steps:
      - name: Deploy selected static reports
        id: deployment
        uses: actions/deploy-pages@v4

```

## 9. Enable and test Actions deliberately

1. Commit the starter and workflow to your repository's default branch.
2. Check repository **Settings ? Actions** permits the workflow and the intended write permission. If protected-branch policy prevents direct commits, adapt to a reviewed PR-based publisher instead.
3. Open **Actions ? Weekly literature watch ? Run workflow** on the default branch.
4. Select `fixture` first. Inspect logs, artifacts and the durable commit.
5. Select `live` after checking the query. Leave Pages off initially.
6. To schedule `agent`, add repository secret `OPENAI_API_KEY`, choose the model in config, and set repository variable `WATCH_MODE=agent`. Otherwise scheduled runs default to `live`.

Fixture dispatch writes labelled demonstration reports; do not mistake those for scientific updates. Live state is separate and the next live run replaces the latest fixture report.

No live workflow has been enabled by the instructor's creation of these files. The learner must intentionally install and activate the template.

**Exercise:** trigger a repeat fixture run and inspect why the event ledger does not double. Do not publish confidential manuscripts in this demonstration.

## 10. Host a selected report with GitHub Pages

The runner writes escaped static HTML in `docs/index.html`; it does not serve a Python backend. The sample website is a simple report viewer, not a full application.

After checking every item intended for publication:

1. In repository **Settings ? Pages**, choose **GitHub Actions** as the publishing source.
2. Set repository variable `PUBLISH_PAGES=true`.
3. Manually run the workflow. The optional Pages artifact/deployment jobs publish `docs/`.
4. Open the deployment URL shown in the job/environment and verify the report/run ID.

Pages availability depends on repository visibility and plan. A private repository does not by itself guarantee a private website: verify site access before publishing research content. Keep unpublished material in the repository/authorized app connection if public hosting is inappropriate.

The combined workflow deploys explicitly; it does not rely on a second push-triggered workflow being launched by its bot commit.

[GitHub publishing-source guidance](https://docs.github.com/en/pages/getting-started-with-github-pages/configuring-a-publishing-source-for-your-github-pages-site)

For this activity, GitHub Pages is the chosen host. No other website platform or hosting provider is needed.

## 11. Three months later: prepare for reviewer comments

Read `reports/review_period.md`. It accumulates observations and dated briefings from saved state since the configured submission date. One paper can have several revision events; observations are not a count of independent studies.

Ask your existing research agent:

```text
Read the project brief and review-period ledger. Group related records by
scientific theme and identify which supplied records may affect contribution,
baselines, data or validation. Cite IDs; distinguish abstract-level suggestions
from verified findings. Include capped/missed-run coverage limitations.
Propose up to three papers for full-text review before manuscript revision.
Do not claim this is all related work. Do not change the submitted paper yet.
```

Human review then determines actual manuscript edits. The watch cannot know when reviews arrive or what reviewers say unless you supply that information.

Use the Week 6A GitHub/phone handoff to discuss the reports. Ask what changed during the review interval and which claims require rechecking. For Claude Projects, refresh selected GitHub content before relying on it.

## 12. Limitations, recovery and maintenance

| Issue | Response |
|---|---|
| Empty results | Inspect query/coverage; do not infer absence of related work |
| Result cap reached | Preserve warning; narrow or backfill deliberately |
| API/network failure | Inspect failed run; saved watermark should remain unchanged |
| Push rejected | Recover artifact; reconcile Git history without force-pushing |
| Scheduled job missed | Review run history; overlap is limited and not exhaustive |
| Model budget exhausted | Keep metadata watch; review pending records separately |
| Scope changes | New output history or explicit migration |
| Website stale | Check deployment log and report ID |

State committed to Git is the durable record. Artifacts have retention limits and are a recovery aid, not the sole three-month archive. Keep the config and query in each run's version history.

Stop recurring work using the workflow's Disable option, and turn off Pages separately if desired. API keys can be revoked independently.

**Exit exercise, before presentations:** identify a false positive, a possible missed paper, a cost control and one human decision. Explain why a weekly AI briefing does not establish scientific validation.

## Instructor notes and next extensions

The shipped starter is tested offline. Live arXiv access, a paid model call, Actions execution and Pages deployment require separate account/network tests; do not describe them as already verified.

Suggested extensions after the mini-example: add a journal metadata source with explicit coverage; monitor releases of a specified dependency; follow selected citation links; or turn a relevance recommendation into a human-approved reading queue. Preserve source-specific identifiers and dates rather than merging unlike events indiscriminately.

This notebook adds no new assessment weights. Keep Assignment 3 presentations as the final activity.

Documentation checked October 5, 2026. Starter scripts and workflow templates are supplied beside the notebook for inspection and adaptation.